In [ ]:
!pip install pyspark delta-spark --break-system-packages -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.7 MB/s eta 0:00:00


In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = SparkSession.builder.appName("DeltaAssignment") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

**Step 1:- Load dataset.**

In [ ]:
df = spark.read.csv("data/Sample - Superstore.csv", header=True, inferSchema=True)
df.show(5)
df.count()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

9994

**Step 2:- Make Customer Table**

In [ ]:
customers = df.select(F.col("Customer ID").alias("Customer_ID"),
                      F.col("Customer Name").alias("Customer_Name"),
                      "Segment", "City", "State", "Region") \
              .dropDuplicates(["Customer_ID"])

customers.show(5)
customers.count()

+-----------+-------------+--------+-----------+--------------+-------+
|Customer_ID|Customer_Name| Segment|       City|         State| Region|
+-----------+-------------+--------+-----------+--------------+-------+
|   AA-10315|   Alex Avila|Consumer|Minneapolis|     Minnesota|Central|
|   AA-10375| Allen Armold|Consumer|       Mesa|       Arizona|   West|
|   AA-10480| Andrew Allen|Consumer|    Concord|North Carolina|  South|
|   AA-10645|Anna Andreadi|Consumer|    Chester|  Pennsylvania|   East|
|   AB-10015|Aaron Bergman|Consumer|    Seattle|    Washington|   West|
+-----------+-------------+--------+-----------+--------------+-------+
only showing top 5 rows


793

In [ ]:
# check nulls
customers.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in customers.columns]).show()

+-----------+-------------+-------+----+-----+------+
|Customer_ID|Customer_Name|Segment|City|State|Region|
+-----------+-------------+-------+----+-----+------+
|          0|            0|      0|   0|    0|     0|
+-----------+-------------+-------+----+-----+------+



In [ ]:
# drop nulls
customers_clean = customers.dropna(subset=["Customer_ID", "Customer_Name"]) \
                            .dropDuplicates(["Customer_ID"])

customers_clean.count()

793

**Step 4:- Save as Delta Table**

In [ ]:
customers_clean.write.format("delta").mode("overwrite").saveAsTable("customer_master")

spark.sql("SELECT * FROM customer_master LIMIT 5").show()

+-----------+-------------+--------+-----------+--------------+-------+
|Customer_ID|Customer_Name| Segment|       City|         State| Region|
+-----------+-------------+--------+-----------+--------------+-------+
|   AA-10315|   Alex Avila|Consumer|Minneapolis|     Minnesota|Central|
|   AA-10375| Allen Armold|Consumer|       Mesa|       Arizona|   West|
|   AA-10480| Andrew Allen|Consumer|    Concord|North Carolina|  South|
|   AA-10645|Anna Andreadi|Consumer|    Chester|  Pennsylvania|   East|
|   AB-10015|Aaron Bergman|Consumer|    Seattle|    Washington|   West|
+-----------+-------------+--------+-----------+--------------+-------+



**Step 5:- Create New (Incremental) Data**

In [ ]:
# take 50 old customers, change their segment (simulate update)
updated = customers_clean.limit(50).withColumn("Segment", F.lit("Corporate"))

# add 20 brand new customers
new_customers = spark.createDataFrame([
    (f"NEW-{i}", f"New Customer {i}", "Consumer", "Jaipur", "Rajasthan", "West") for i in range(1, 21)
], ["Customer_ID", "Customer_Name", "Segment", "City", "State", "Region"])

incremental = updated.union(new_customers)
incremental.createOrReplaceTempView("incremental_data")

incremental.show(100)

+-----------+--------------------+---------+----------------+--------------+-------+
|Customer_ID|       Customer_Name|  Segment|            City|         State| Region|
+-----------+--------------------+---------+----------------+--------------+-------+
|   AA-10315|          Alex Avila|Corporate|     Minneapolis|     Minnesota|Central|
|   AA-10375|        Allen Armold|Corporate|            Mesa|       Arizona|   West|
|   AA-10480|        Andrew Allen|Corporate|         Concord|North Carolina|  South|
|   AA-10645|       Anna Andreadi|Corporate|         Chester|  Pennsylvania|   East|
|   AB-10015|       Aaron Bergman|Corporate|         Seattle|    Washington|   West|
|   AB-10060|     Adam Bellavance|Corporate|   New York City|      New York|   East|
|   AB-10105|       Adrian Barton|Corporate|         Phoenix|       Arizona|   West|
|   AB-10150|         Aimee Bixby|Corporate|      Long Beach|      New York|   East|
|   AB-10165|         Alan Barnes|Corporate|     Los Angeles|    

In [ ]:
# Save customer_master.csv (original clean data, before merge)
customers_clean.toPandas().to_csv("data/customer_master.csv", index=False)

# Save customer_incremental.csv (new/updated data)
incremental.toPandas().to_csv("data/customer_incremental.csv", index=False)

**Step 6:- MERGE (update old + insert new)**

In [ ]:
spark.sql("""
MERGE INTO customer_master AS t
USING incremental_data AS s
ON t.Customer_ID = s.Customer_ID

WHEN MATCHED THEN
UPDATE SET t.Customer_Name = s.Customer_Name, t.Segment = s.Segment, t.City = s.City, t.State = s.State, t.Region = s.Region

WHEN NOT MATCHED THEN
INSERT (Customer_ID, Customer_Name, Segment, City, State, Region) VALUES (s.Customer_ID, s.Customer_Name, s.Segment, s.City, s.State, s.Region)
""")

spark.sql("SELECT * FROM customer_master WHERE Segment = 'Corporate'").show()

+-----------+--------------------+---------+---------------+--------------+-------+
|Customer_ID|       Customer_Name|  Segment|           City|         State| Region|
+-----------+--------------------+---------+---------------+--------------+-------+
|   AA-10315|          Alex Avila|Corporate|    Minneapolis|     Minnesota|Central|
|   AA-10375|        Allen Armold|Corporate|           Mesa|       Arizona|   West|
|   AA-10480|        Andrew Allen|Corporate|        Concord|North Carolina|  South|
|   AA-10645|       Anna Andreadi|Corporate|        Chester|  Pennsylvania|   East|
|   AB-10015|       Aaron Bergman|Corporate|        Seattle|    Washington|   West|
|   AB-10060|     Adam Bellavance|Corporate|  New York City|      New York|   East|
|   AB-10105|       Adrian Barton|Corporate|        Phoenix|       Arizona|   West|
|   AB-10150|         Aimee Bixby|Corporate|     Long Beach|      New York|   East|
|   AB-10165|         Alan Barnes|Corporate|    Los Angeles|    California| 

**Step 7:- Validate**

In [ ]:
# total rows
spark.sql("SELECT COUNT(*) FROM customer_master").show()

+--------+
|count(1)|
+--------+
|     813|
+--------+



In [ ]:
# see the change history of the delta table
spark.sql("DESCRIBE HISTORY customer_master").show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      4|2026-07-12 09:24:...|  NULL|    NULL|               MERGE|{predicate -> ["(...|NULL|    NULL|     NULL|          3|  Serializable|        false|{numTargetRowsCop...|        NULL|Apache-Spark/4.0....|
|      3|2026-07-12 09:21:...|  NULL|    NULL|CREATE OR REPLACE...|{isV1SaveAsTableO...|NULL|    NULL|     NULL|          2|  Serializable|        false|{numFiles -

**Step 8:- Final Output**

In [ ]:
spark.sql("SELECT * FROM customer_master ORDER BY `Customer_ID` LIMIT 20").show()

+-----------+--------------------+---------+---------------+--------------+-------+
|Customer_ID|       Customer_Name|  Segment|           City|         State| Region|
+-----------+--------------------+---------+---------------+--------------+-------+
|   AA-10315|          Alex Avila|Corporate|    Minneapolis|     Minnesota|Central|
|   AA-10375|        Allen Armold|Corporate|           Mesa|       Arizona|   West|
|   AA-10480|        Andrew Allen|Corporate|        Concord|North Carolina|  South|
|   AA-10645|       Anna Andreadi|Corporate|        Chester|  Pennsylvania|   East|
|   AB-10015|       Aaron Bergman|Corporate|        Seattle|    Washington|   West|
|   AB-10060|     Adam Bellavance|Corporate|  New York City|      New York|   East|
|   AB-10105|       Adrian Barton|Corporate|        Phoenix|       Arizona|   West|
|   AB-10150|         Aimee Bixby|Corporate|     Long Beach|      New York|   East|
|   AB-10165|         Alan Barnes|Corporate|    Los Angeles|    California| 

In [ ]:
print("Before merge:", customers_clean.count())
print("Incremental data:", incremental.count())
print("After merge:", spark.sql("SELECT COUNT(*) FROM customer_master").collect()[0][0])

Before merge: 793
Incremental data: 70
After merge: 813
